# Детекция ботов-сборщиков данных — решение

Ноутбук проходит весь путь от сырых событий до `submission.csv`. Логика лежит в модулях `features.py`
(очистка и признаки) и `train.py` (валидация и модели), ноутбук вызывает их по шагам и объясняет каждый шаг.
Запуск ноутбука целиком даёт **ровно тот же** `submission.csv`, что отправлен на платформу (≈2 мин на ноутбуке).

Разведочный анализ, из которого выведены признаки, — `01_eda.ipynb`; полная история экспериментов — `EXPERIMENTS.md`;
подробный отчёт — `README.md`.

## Подход в двух словах

**Задача.** По событиям куки внутри суточного окна выдать score «насколько это сервис сбора данных».
Метрика — Precision при Recall ≥ 70%: важен только порядок кук, одинаковые score отмечаются группой.

**Данные.** Файл событий перемешан по времени, содержит ~4.9 тыс. полных дублей, `platform` записан в разных
регистрах, а ~40 тыс. событий лежат *после* окна наблюдения. Очистка: сортировка, удаление дублей (доля дублей —
признак), нормализация, и **только события внутри `[window_start_ts, window_end_ts)`** — признаки не видят будущего.

**Признаки (128)** — поведение, а не «отпечатки» сервисов:
* **курсор** — главный сигнал: люди кликают равномерно по экрану, боты «блуждают» с малым шагом, упираются в края
  экрана и часто вообще не передают координаты;
* **тайминг** — квантили интервалов между событиями внутри сессии (боты заметно быстрее);
* **широта обхода** — города, категории, глубина листания выдачи;
* действия, время суток, возраст куки;
* **разобранный User-Agent** — тип клиента (приложение / браузер), ОС, противоречия с платформой; сырая строка UA
  не используется — это идентификатор сервиса, а не поведение;
* **популярность просмотренных объявлений** среди других кук за последние сутки до конца окна.

**Валидация.** Test идёт по времени сразу после train, поэтому валидация временная: 4 фолда, обучение только на
прошлых днях. Метрика считается по склеенным out-of-fold предсказаниям, плюс бутстреп-интервал.

**Модель.** Ранговое среднее LightGBM + CatBoost + ExtraTrees; ничьи в сумме рангов разбиваются средней
вероятностью моделей (метрика штрафует одинаковые score).

**Результат.** Baseline из quickstart 0.093 → **0.833 на CV** (CI90 0.778–0.861) → **0.842 на скрытом тесте**.

**Что нашли по дороге.** Первая версия популярности объявлений заглядывала в будущее (+0.10 фиктивного качества),
вторая давала сдвиг train/test (adversarial AUC 0.886) — обе исправлены. Проверки устойчивости: train и test
неотличимы (adversarial AUC 0.51); на *незнакомом* типе ботов качество заметно ниже — главное ограничение.

## Где лежит код

Ноутбук — точка входа: он вызывает функции из двух модулей, чтобы логика жила в одном месте и не дублировалась.
Весь код прокомментирован; ниже — карта, где искать каждый шаг.

| Шаг | Файл | Функция | Что делает |
|---|---|---|---|
| загрузка | `features.py` | `load_data` | читает train / test / events, парсит даты |
| очистка и окно | `features.py` | `clean_events` | нормализация, удаление дублей, только события внутри окна, сортировка |
| тайминг и сессии | `features.py` | `volume_timing` | сессии (разрыв > 30 мин), квантили интервалов, пиковая активность |
| действия | `features.py` | `event_mix` | доли типов событий, биграммы переходов |
| контент и поиск | `features.py` | `content`, `search` | города, категории, глубина и линейность листания выдачи |
| курсор | `features.py` | `pointer` | шаг между точками, автокорреляция, упоры в края экрана, пропуски |
| User-Agent | `features.py` | `_parse_ua`, `ua_platform` | тип клиента, ОС, версия, противоречия с платформой |
| время, возраст | `features.py` | `hours`, `age` | время суток, возраст куки, доля дублей |
| популярность объявлений | `features.py` | `cross_cookie` | сколько кук открыли объявление за сутки до конца окна (без будущего) |
| сборка признаков | `features.py` | `build_features` | всё вместе для train и test |
| выбор признаков | `train.py` | `prepare`, `EXCLUDED_GROUPS`, `UA_PARSED` | финальный набор из 128 признаков |
| модели | `train.py` | `fit_predict` | LightGBM, CatBoost, ExtraTrees и их ранговое среднее |
| валидация | `train.py` | `time_cv`, `bootstrap_ci` | временные фолды, OOF-метрика, доверительный интервал |
| все эксперименты | `train.py` | `main` | `python3 train.py`: baseline, контрольные варианты, ablation, сабмит |
| проверки устойчивости | `validate.py` | `adversarial`, `cluster_bots`, `loco`, `tune` | сдвиг train/test, перенос на незнакомый тип ботов, тюнинг |
| разведка данных | `01_eda.ipynb` | — | наблюдения, из которых выведены признаки |

## 1. Окружение и настройки

In [1]:
import sys, hashlib
import numpy as np
import pandas as pd

from features import load_data, clean_events, feature_group
import train as T   # функции валидации и моделей + все константы решения

print('Python', sys.version.split()[0])
print('seed:', T.SEED, '| сидов в CV:', T.CV_SEEDS, '| сидов в финале:', T.FINAL_SEEDS)
print('временные фолды (валидация):', T.TIME_FOLDS)
print('финальная модель:', T.FINAL_MODEL)

Python 3.13.10
seed: 42 | сидов в CV: 3 | сидов в финале: 5
временные фолды (валидация): [('2026-04-12', '2026-04-14'), ('2026-04-14', '2026-04-16'), ('2026-04-16', '2026-04-18'), ('2026-04-18', '2026-04-20')]
финальная модель: blend


## 2. Данные и очистка событий

`clean_events` (в `features.py`) делает всю гигиену: удаляет `eid` (дублирует `event_name`), нормализует `platform`
и текстовые поля, удаляет полные дубли, оставляет только события внутри окна и сортирует по времени внутри куки.

In [2]:
DATA = 'data'
train, test, events = load_data(DATA)
meta = pd.concat([train.drop(columns='target'), test], ignore_index=True)
ev, aux = clean_events(events, meta)

print(f'train {train.shape}, test {test.shape}, доля ботов {train.target.mean():.3f}')
print(f'событий всего {len(events):,}, полных дублей удалено {int(aux.n_dup.sum()):,}, '
      f'в окнах {len(ev):,} (остальное — после окна, отброшено)')
print('platform после нормализации:', sorted(ev.platform.dropna().unique()))

train (11091, 5), test (4909, 4), доля ботов 0.081
событий всего 328,905, полных дублей удалено 4,868, в окнах 283,830 (остальное — после окна, отброшено)
platform после нормализации: ['android', 'desktop', 'ios', 'web']


## 3. Признаки

`T.prepare()` строит все признаки для train и test за один проход (таргет не используется) и возвращает финальный
набор колонок. Префикс имени признака — его группа. Группы `w` (время на странице) и `leak` (версия популярности
объявлений с заглядыванием в будущее) строятся только для экспериментов и в модель не входят.

In [3]:
train, test, Xtr, Xte, y, dates, all_cols, final_cols = T.prepare(DATA)

groups = {'p': 'курсор', 't': 'тайминг', 'c': 'контент', 's': 'поиск', 'v': 'объём/сессии', 'e': 'действия',
          'h': 'время суток', 'a': 'возраст куки', 'u': 'разобранный UA', 'x': 'популярность объявлений'}
pd.Series([feature_group(c) for c in final_cols]).map(groups).value_counts().rename('признаков').to_frame()

данные: {'events_raw': 328905, 'events_in_windows': 283830, 'dups_removed': 4868, 'prior_events': 0, 'text_values_merged': {'search_query': 0, 'item_category': 0, 'item_location': 0, 'seller_type': 0}, 'item_pop_lookback': '1 days 00:00:00'} | признаков: всего 153, в финальной модели 128


,признаков
действия,38
тайминг,17
курсор,16
разобранный UA,13
контент,10
поиск,10
объём/сессии,8
время суток,7
возраст куки,5
популярность объявлений,4


In [4]:
# пример: признаки курсора и тайминга для нескольких кук train
Xtr.assign(target=y)[['target', 'p_share_missing_web', 'p_step_mean', 'p_share_border',
                      't_dt_q50', 't_dt_q90', 'c_n_loc', 's_page_max', 'u_family']].head(8)

/var/folders/2v/5nxmvvpd5n54qt8tt30brbbm0000gn/T/ipykernel_7114/2014850748.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  Xtr.assign(target=y)[['target', 'p_share_missing_web', 'p_step_mean', 'p_share_border',


,target,p_share_missing_web,p_step_mean,p_share_border,t_dt_q50,t_dt_q90,c_n_loc,s_page_max,u_family
0,0,NaN,NaN,NaN,21.0,59.0,5,6.0,avito_app
1,0,0.073171,660.463913,0.0,50.5,82.7,12,8.0,yandex
2,0,0.027778,967.224037,0.0,21.0,68.2,9,10.0,chrome
3,0,1.000000,NaN,NaN,52.0,70.6,4,2.0,chrome
4,0,0.035714,657.930597,0.0,46.5,181.3,6,7.0,chrome
5,0,0.111111,775.660995,0.0,32.0,148.4,5,16.0,chrome
6,0,NaN,NaN,NaN,63.5,196.2,5,1.0,chrome
7,0,NaN,NaN,NaN,29.5,1395.7,8,8.0,chrome


## 4. Валидация: baseline против финальной модели

`T.time_cv` обучает модель на днях до начала каждого валидационного блока и предсказывает следующий блок.
Главная цифра — P@R70 по склеенным out-of-fold предсказаниям; `T.bootstrap_ci` даёт 90%-интервал.
Метрика — официальная `metric.precision_at_recall`.

In [5]:
res_base, _ = T.time_cv(Xtr, y, dates, ['v_n_events', 'c_n_item'], model='rf')      # baseline из quickstart
res_final, oof = T.time_cv(Xtr, y, dates, final_cols, model=T.FINAL_MODEL)            # финальная модель
m = oof.notna().to_numpy()
lo, hi = T.bootstrap_ci(y[m], oof[m].to_numpy())

pd.DataFrame([res_base, res_final], index=['baseline (RF, 2 признака)', 'финал (ансамбль, 128 признаков)']) \
  .drop(columns='p_at_r70_fold_mean').round(4).assign(ci90=['', f'{lo:.3f}–{hi:.3f}'])

,p_at_r70_oof,p_at_r70_folds,pr_auc_oof,roc_auc_oof,ci90
"baseline (RF, 2 признака)",0.0930,0.084 0.108 0.089 0.098,0.2022,0.6088,
"финал (ансамбль, 128 признаков)",0.8325,0.850 0.828 0.864 0.786,0.8056,0.9451,0.778–0.861


## 5. Все эксперименты

Полная таблица — результат `python3 train.py` (сохраняется в `results/experiments.csv`): контрольные варианты
(с утечкой, без UA, без популярности объявлений, другие модели) и ablation — удаление каждой группы признаков.

In [6]:
from pathlib import Path
p = Path('results/experiments.csv')
if p.exists():
    display(pd.read_csv(p)[['experiment', 'n_features', 'p_at_r70_oof', 'p_at_r70_folds', 'pr_auc_oof', 'roc_auc_oof']])
else:
    print('запустите python3 train.py, чтобы получить полную таблицу экспериментов')

,experiment,n_features,p_at_r70_oof,p_at_r70_folds,pr_auc_oof,roc_auc_oof
0,baseline_rf_quickstart,2,0.0930,0.084 0.108 0.089 0.098,0.2022,0.6088
1,baseline_lgb_quickstart,2,0.1125,0.110 0.105 0.128 0.100,0.1911,0.6568
2,blend_final,128,0.8325,0.850 0.828 0.864 0.786,0.8056,0.9451
3,blend_without_itempop,124,0.8222,0.835 0.826 0.841 0.776,0.7999,0.9427
4,lgb_final_features,128,0.8005,0.816 0.826 0.781 0.767,0.8030,0.9453
5,cat_final_features,128,0.8215,0.848 0.856 0.820 0.776,0.8040,0.9432
6,lgb_behavior_only,111,0.7872,0.832 0.798 0.774 0.750,0.7920,0.9409
7,blend_behavior_only,111,0.7967,0.851 0.805 0.812 0.750,0.7957,0.9419
8,lgb_with_ua_popularity,129,0.7933,0.816 0.838 0.763 0.773,0.8040,0.9445
9,lgb_itempop_leaky,128,0.8907,0.923 0.896 0.909 0.840,0.8294,0.9508


## 6. Финальная модель и `submission.csv`

Обучение на всём train, предсказание test. Детерминизм: фиксированные сиды, `deterministic=True` в LightGBM,
фиксированное число потоков CatBoost, однопоточное предсказание ExtraTrees — повторный запуск даёт побайтно тот же файл.

In [7]:
score, imp = T.fit_predict(Xtr[final_cols], y, Xte[final_cols], T.FINAL_MODEL, T.FINAL_SEEDS)
sub = pd.DataFrame({'cookie_id': test['cookie_id'].to_numpy(), 'score': score})

# проверки формата
assert len(sub) == len(test) and sub.cookie_id.is_unique and set(sub.cookie_id) == set(test.cookie_id)
assert sub.score.notna().all() and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)

print(f'submission.csv: {len(sub)} строк, уникальных score: {sub.score.nunique()}')
print('md5:', hashlib.md5(open('submission.csv', 'rb').read()).hexdigest())
sub.head()

submission.csv: 4909 строк, уникальных score: 4909
md5: 5eda6e8f43b2f4d715c53f9b5ea1f5bb


,cookie_id,score
0,ck_315fb710a0e371e7,0.034359
1,ck_a76ee3b3e3e522fd,0.891219
2,ck_94c9a4d382689e82,0.699870
3,ck_8eaf9509ad9462a0,0.145006
4,ck_9a88a5a989cb5bc6,0.343586


In [8]:
# вклад признаков в LightGBM-часть ансамбля (доля gain)
(imp / imp.sum()).sort_values(ascending=False).head(15).round(3).rename('gain_share').to_frame()

,gain_share
t_dt_q90,0.118
p_std_x,0.069
p_share_border,0.052
t_share_dt_le30,0.042
t_dt_q75,0.039
t_dt_q50,0.035
c_cat_entropy,0.031
t_logdt_std,0.031
c_loc_entropy,0.028
c_n_cat,0.027


## 7. Ограничения

* Модель хорошо ловит **известные** сервисы; если тип ботов исключить из обучения (leave-one-cluster-out,
  `validate.py`), ROC-AUC на нём падает до 0.66–0.81. Новый сервис потребует дообучения.
* На мобильных платформах координат курсора нет, и recall там ниже (android ~0.5 против ~0.8 на web).
* Куки с 1–3 событиями почти не распознаются — информации слишком мало.
* Популярность объявлений считается по событиям всех кук train+test (трансдуктивно, без таргета и без будущего).
* Финальный вариант выбран с учётом второго сабмита, поэтому честная ожидаемая оценка — CV 0.833, а не 0.842.

Подробности, таблицы и история решений — `README.md` и `EXPERIMENTS.md`.